In [17]:
import os
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())
gemini_api_key = os.environ.get("GEMINI_API_KEY")
gemini_model = os.environ.get("GEMINI_MODEL")

In [18]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("data/be-good.txt")
documents = loader.load()

In [19]:
data = documents[0].page_content

## Character text splitter

In [20]:
from langchain_text_splitters import CharacterTextSplitter

text_splitter = CharacterTextSplitter(separator="\n\n", chunk_size=1000, chunk_overlap=200, length_function= len, is_separator_regex=False)
split_documents = text_splitter.split_documents(documents)

In [21]:
text = text_splitter.create_documents([data])

In [22]:
text

[Document(metadata={}, page_content='Be good'),
 Document(metadata={}, page_content='April 2008(This essay is derived from a talk at the 2008 Startup School.)About a month after we started Y Combinator we came up with the\nphrase that became our motto: Make something people want.  We\'ve\nlearned a lot since then, but if I were choosing now that\'s still\nthe one I\'d pick.Another thing we tell founders is not to worry too much about the\nbusiness model, at least at first.  Not because making money is\nunimportant, but because it\'s so much easier than building something\ngreat.A couple weeks ago I realized that if you put those two ideas\ntogether, you get something surprising.  Make something people want.\nDon\'t worry too much about making money.  What you\'ve got is a\ndescription of a charity.When you get an unexpected result like this, it could either be a\nbug or a new discovery.  Either businesses aren\'t supposed to be\nlike charities, and we\'ve proven by reductio ad absurdum

In [23]:
text[1]

Document(metadata={}, page_content='April 2008(This essay is derived from a talk at the 2008 Startup School.)About a month after we started Y Combinator we came up with the\nphrase that became our motto: Make something people want.  We\'ve\nlearned a lot since then, but if I were choosing now that\'s still\nthe one I\'d pick.Another thing we tell founders is not to worry too much about the\nbusiness model, at least at first.  Not because making money is\nunimportant, but because it\'s so much easier than building something\ngreat.A couple weeks ago I realized that if you put those two ideas\ntogether, you get something surprising.  Make something people want.\nDon\'t worry too much about making money.  What you\'ve got is a\ndescription of a charity.When you get an unexpected result like this, it could either be a\nbug or a new discovery.  Either businesses aren\'t supposed to be\nlike charities, and we\'ve proven by reductio ad absurdum that one\nor both of the principles we began wit

## Recursive text splitter

In [24]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

recursive_text_splitter = RecursiveCharacterTextSplitter(chunk_size=26, chunk_overlap=4)
recursive_split_documents = recursive_text_splitter.split_text(data)

In [25]:
recursive_split_documents

['Be good',
 'April 2008(This essay is',
 'is derived from a talk at',
 'at the 2008 Startup',
 'School.)About a month',
 'after we started Y',
 'Y Combinator we came up',
 'up with the',
 'phrase that became our',
 'our motto: Make something',
 "people want.  We've",
 'learned a lot since then,',
 'but if I were choosing',
 "now that's still",
 "the one I'd pick.Another",
 'thing we tell founders is',
 'is not to worry too much',
 'about the',
 'business model, at least',
 'at first.  Not because',
 'making money is',
 'unimportant, but because',
 "it's so much easier than",
 'building something',
 'great.A couple weeks ago',
 'ago I realized that if',
 'if you put those two',
 'two ideas',
 'together, you get',
 'get something surprising.',
 'Make something people',
 'want.',
 "Don't worry too much",
 'about making money.  What',
 "you've got is a",
 'description of a',
 'a charity.When you get an',
 'an unexpected result like',
 'this, it could either be',
 'be a',
 'bug or a new di

## Embeddings

Transform the small parts of text in numbers (vectors) that are easily stored and searched by vector databases.

In [26]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embedding_model = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2")

chain_of_text =[
    'Hii, how are you?',
    'I am fine, thank you!',
    'What about you?',
    'I am doing well, thanks for asking!'
]

embedding = embedding_model.embed_documents(chain_of_text)



In [27]:
embedding

[[-0.015776739,
  0.006173067,
  0.006561323,
  -0.013945689,
  -0.00863999,
  0.009912885,
  0.00052321376,
  0.007880755,
  0.016365616,
  -0.036352098,
  -0.02356191,
  0.0055732,
  -0.008007495,
  -0.020473158,
  0.014952451,
  -0.020122774,
  0.028675064,
  -0.019130504,
  0.006670092,
  -0.004269263,
  -0.00043835465,
  0.0030546752,
  0.009734591,
  0.01399162,
  0.0011210942,
  0.006092345,
  0.010892852,
  -0.0026395,
  -0.019971812,
  0.12308518,
  -0.014689239,
  -0.008800771,
  -0.0019125647,
  -0.006919649,
  -0.00375154,
  0.016502392,
  0.014135033,
  -0.017045792,
  0.011278437,
  0.024570532,
  0.037098825,
  0.0016585694,
  -0.0067888265,
  0.0029495815,
  -0.010366827,
  0.002217112,
  -0.027497564,
  -0.01205808,
  0.0084381625,
  0.019871961,
  -0.010010636,
  -0.0008619,
  -0.0057711015,
  -0.02220042,
  -0.021715224,
  -0.007722135,
  -0.0036131644,
  0.010435821,
  -0.042184047,
  0.016243005,
  -0.010989905,
  0.0014699365,
  0.0120670805,
  0.014437455,
  0.01

In [31]:
len_embedding = len(embedding[0])
len_embedding

3072

In [32]:
for e in embedding:
    print(len(e))

3072
3072
3072
3072


## Vector Stores (aka. Vector Databases)

Store embeddings in a very fast searchable database.

In [34]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_chroma import Chroma


In [36]:
load_documents = TextLoader("data/state_of_the_union.txt")
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(load_documents.load())

In [37]:
chunks

[Document(metadata={'source': 'data/state_of_the_union.txt'}, page_content='Madam Speaker, Madam Vice President, our First Lady and Second Gentleman. Members of Congress and the Cabinet. Justices of the Supreme Court. My fellow Americans.  \n\nLast year COVID-19 kept us apart. This year we are finally together again. \n\nTonight, we meet as Democrats Republicans and Independents. But most importantly as Americans. \n\nWith a duty to one another to the American people to the Constitution. \n\nAnd with an unwavering resolve that freedom will always triumph over tyranny. \n\nSix days ago, Russia’s Vladimir Putin sought to shake the foundations of the free world thinking he could make it bend to his menacing ways. But he badly miscalculated. \n\nHe thought he could roll into Ukraine and the world would roll over. Instead he met a wall of strength he never imagined. \n\nHe met the Ukrainian people. \n\nFrom President Zelenskyy to every Ukrainian, their fearlessness, their courage, their det

In [38]:
len(chunks)

49

In [ ]:
vectorDB = Chroma.from_documents(chunks, embedding_model, persist_directory="./chroma_db")

In [41]:
vectorDB

In [43]:
question = "What did the president say about the John Lewis Voting Rights Act?"
docs = vectorDB.similarity_search(question)

In [45]:
print(docs[0].page_content)

In state after state, new laws have been passed, not only to suppress the vote, but to subvert entire elections. 

We cannot let this happen. 

Tonight. I call on the Senate to: Pass the Freedom to Vote Act. Pass the John Lewis Voting Rights Act. And while you’re at it, pass the Disclose Act so Americans can know who is funding our elections. 

Tonight, I’d like to honor someone who has dedicated his life to serve this country: Justice Stephen Breyer—an Army veteran, Constitutional scholar, and retiring Justice of the United States Supreme Court. Justice Breyer, thank you for your service. 

One of the most serious constitutional responsibilities a President has is nominating someone to serve on the United States Supreme Court. 

And I did that 4 days ago, when I nominated Circuit Court of Appeals Judge Ketanji Brown Jackson. One of our nation’s top legal minds, who will continue Justice Breyer’s legacy of excellence.


## Vector Store as Retriever

Find the embedding that best answers your question.

In [47]:
retriever = vectorDB.as_retriever(search_kwargs={"k": 1})
response = retriever.invoke("what did he say about ketanji brown jackson?")
print(response)

[Document(id='f73ca44c-83f6-4286-ba7d-837450449a70', metadata={'source': 'data/state_of_the_union.txt'}, page_content='And I did that 4 days ago, when I nominated Circuit Court of Appeals Judge Ketanji Brown Jackson. One of our nation’s top legal minds, who will continue Justice Breyer’s legacy of excellence. \n\nA former top litigator in private practice. A former federal public defender. And from a family of public school educators and police officers. A consensus builder. Since she’s been nominated, she’s received a broad range of support—from the Fraternal Order of Police to former judges appointed by Democrats and Republicans. \n\nAnd if we are to advance liberty and justice, we need to secure the Border and fix the immigration system. \n\nWe can do both. At our border, we’ve installed new technology like cutting-edge scanners to better detect drug smuggling.  \n\nWe’ve set up joint patrols with Mexico and Guatemala to catch more human traffickers.  \n\nWe’re putting in place dedi

In [48]:
len(response)

1